# CVRP Customer-to-Vehicle Assignment QUBOs

This notebook builds customer-to-vehicle assignment QUBOs for the same CVRP instance. It covers two versions: an 80-variable soft-capacity assignment QUBO and a 112-variable hard-capacity-slack assignment QUBO. It stops at QUBO conversion. It does not run VQE, QAOA, sampling, or any quantum simulation.


## Why this uses fewer qubits

This formulation does not encode route order. It only decides which vehicle cluster each customer belongs to. With 20 customers and 4 vehicles, the assignment variables are `20 * 4 = 80` binary variables.

The soft-capacity version keeps only those 80 assignment variables and adds quadratic load-balance penalties to the objective. The hard-capacity-slack version adds capacity constraints, and `QuadraticProgramToQubo` introduces 8 slack bits per vehicle because capacity is 231. That gives `80 + 4 * 8 = 112` QUBO variables.

Tradeoff: route ordering is deferred to a later route-second method. This notebook only builds the cluster assignment QUBOs.


In [1]:
from pathlib import Path
import math
import re

import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo


def find_cvrp_dir():
    candidates = [
        Path('cvrp'),
        Path('individual') / 'cvrp',
        Path.cwd() / 'cvrp',
        Path.cwd() / 'individual' / 'cvrp',
    ]
    for candidate in candidates:
        if candidate.exists() and any(candidate.glob('*.vrp')):
            return candidate
    raise FileNotFoundError('Could not find a cvrp folder containing .vrp files.')


def read_cvrp_instance(file_path):
    file_path = Path(file_path)
    metadata = {}
    coords = {}
    demands = {}
    depots = []
    section = None

    for raw_line in file_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line == 'EOF':
            break
        if line in {'NODE_COORD_SECTION', 'DEMAND_SECTION', 'DEPOT_SECTION'}:
            section = line
            continue

        if section == 'NODE_COORD_SECTION':
            node, x_coord, y_coord = line.split()[:3]
            coords[int(node)] = (float(x_coord), float(y_coord))
            continue

        if section == 'DEMAND_SECTION':
            node, demand = line.split()[:2]
            demands[int(node)] = int(float(demand))
            continue

        if section == 'DEPOT_SECTION':
            node = int(line.split()[0])
            if node != -1:
                depots.append(node)
            continue

        if ':' in line:
            key, value = line.split(':', 1)
            metadata[key.strip()] = value.strip().strip('"')

    if not depots:
        raise ValueError(f'No depot found in {file_path}')
    if set(coords) != set(demands):
        raise ValueError('Coordinate and demand sections have different node ids.')

    depot = depots[0]
    nodes = sorted(coords)
    customers = [node for node in nodes if node != depot]
    return {
        'path': file_path,
        'metadata': metadata,
        'coords': coords,
        'demands': demands,
        'depots': depots,
        'depot': depot,
        'nodes': nodes,
        'customers': customers,
    }


def parse_vehicle_count(instance, default=1):
    name = instance['metadata'].get('NAME', instance['path'].stem)
    match = re.search(r'-k(\d+)', name)
    return int(match.group(1)) if match else default


def euclidean_distance(instance, a, b):
    ax, ay = instance['coords'][a]
    bx, by = instance['coords'][b]
    return int(round(math.hypot(ax - bx, ay - by)))


def bit_width_from_qubo(qubo):
    return qubo.get_num_vars()


def print_instance_summary(instance):
    capacity = int(instance['metadata']['CAPACITY'])
    parsed_vehicles = parse_vehicle_count(instance, default=len(instance['customers']))
    total_demand = sum(instance['demands'][node] for node in instance['customers'])
    print(f"Instance: {instance['metadata'].get('NAME', instance['path'].stem)}")
    print(f"Comment: {instance['metadata'].get('COMMENT', '')}")
    print(f"Customers: {len(instance['customers'])}")
    print(f"Depot: {instance['depot']}")
    print(f"Parsed vehicles: {parsed_vehicles}")
    print(f"Capacity: {capacity}")
    print(f"Total demand: {total_demand}")
    print(f"Demand / capacity: {total_demand / capacity:.2f}")


In [2]:
INSTANCE_NAME = 'XSH-n20-k4-01.vrp'
CVRP_DIR = find_cvrp_dir()
instance = read_cvrp_instance(CVRP_DIR / INSTANCE_NAME)
parsed_vehicle_count = parse_vehicle_count(instance, default=len(instance['customers']))
vehicle_capacity = int(instance['metadata']['CAPACITY'])

print(f"CVRP directory: {CVRP_DIR.resolve()}")
print_instance_summary(instance)


CVRP directory: /Users/monitsharma/SMU-Quantum-Repos/autoqresearch/individual/cvrp
Instance: XSH-n20-k4-01
Comment: Queiroga and Uchoa (2024); Optimal cost: 646
Customers: 20
Depot: 1
Parsed vehicles: 4
Capacity: 231
Total demand: 924
Demand / capacity: 4.00


## Assignment variable setup

`z_customer_vehicle = 1` means a customer is assigned to a vehicle cluster. Each customer must be assigned to exactly one vehicle. We also add a simple compactness objective: customers that are far apart are discouraged from being assigned to the same vehicle.


In [3]:
def assignment_variable_name(customer, vehicle):
    return f'z_{customer}_{vehicle}'


def add_assignment_variables(qp, customers, num_vehicles):
    for customer in customers:
        for vehicle in range(num_vehicles):
            qp.binary_var(name=assignment_variable_name(customer, vehicle))


def add_each_customer_once_constraints(qp, customers, num_vehicles):
    for customer in customers:
        qp.linear_constraint(
            linear={assignment_variable_name(customer, vehicle): 1 for vehicle in range(num_vehicles)},
            sense='==',
            rhs=1,
            name=f'assign_customer_{customer}',
        )


def compactness_quadratic_terms(instance, num_vehicles, scale=1.0):
    customers = instance['customers']
    quadratic = {}
    for vehicle in range(num_vehicles):
        for left_index, left in enumerate(customers):
            for right in customers[left_index + 1:]:
                distance = scale * euclidean_distance(instance, left, right)
                key = (assignment_variable_name(left, vehicle), assignment_variable_name(right, vehicle))
                quadratic[key] = quadratic.get(key, 0.0) + distance
    return quadratic


def merge_quadratic(target, source):
    for key, value in source.items():
        target[key] = target.get(key, 0.0) + value


## Soft-capacity assignment QUBO

This version does not add capacity as a hard linear constraint. Instead, it adds a quadratic load penalty for each vehicle: `(load_k - capacity)^2`. Since this instance has total demand equal to `4 * capacity`, this penalty encourages every vehicle cluster to land near capacity while keeping the variable count at 80 before and after QUBO conversion.


In [4]:
def soft_capacity_load_penalty(instance, num_vehicles, penalty):
    customers = instance['customers']
    capacity = int(instance['metadata']['CAPACITY'])
    linear = {}
    quadratic = {}
    constant = 0.0

    for vehicle in range(num_vehicles):
        constant += penalty * capacity * capacity
        for customer in customers:
            variable = assignment_variable_name(customer, vehicle)
            demand = instance['demands'][customer]
            linear[variable] = linear.get(variable, 0.0) + penalty * (demand * demand - 2 * capacity * demand)
        for left_index, left in enumerate(customers):
            for right in customers[left_index + 1:]:
                left_var = assignment_variable_name(left, vehicle)
                right_var = assignment_variable_name(right, vehicle)
                value = 2 * penalty * instance['demands'][left] * instance['demands'][right]
                quadratic[(left_var, right_var)] = quadratic.get((left_var, right_var), 0.0) + value

    return constant, linear, quadratic


def build_soft_capacity_assignment_qp(instance, num_vehicles, capacity_penalty=1000.0, compactness_scale=1.0):
    qp = QuadraticProgram('cvrp_soft_capacity_assignment')
    add_assignment_variables(qp, instance['customers'], num_vehicles)
    add_each_customer_once_constraints(qp, instance['customers'], num_vehicles)

    constant, linear, quadratic = soft_capacity_load_penalty(instance, num_vehicles, capacity_penalty)
    merge_quadratic(quadratic, compactness_quadratic_terms(instance, num_vehicles, scale=compactness_scale))
    qp.minimize(constant=constant, linear=linear, quadratic=quadratic)
    return qp


soft_assignment_qp = build_soft_capacity_assignment_qp(instance, parsed_vehicle_count)
soft_assignment_converter = QuadraticProgramToQubo()
soft_assignment_qubo = soft_assignment_converter.convert(soft_assignment_qp)

print('Soft-capacity assignment')
print(f"Original variables: {soft_assignment_qp.get_num_vars()}")
print(f"Original constraints: {len(soft_assignment_qp.linear_constraints)}")
print(f"QUBO variables: {soft_assignment_qubo.get_num_vars()}")
print(f"Converter penalty: {soft_assignment_converter.penalty}")


Soft-capacity assignment
Original variables: 80
Original constraints: 20
QUBO variables: 80
Converter penalty: 4681004997.0


## Hard-capacity assignment QUBO with slack

This version adds capacity as a hard linear inequality for each vehicle: `sum demand_i * z_i_k <= capacity`. Qiskit's QUBO converter binary-expands slack variables for these inequalities. Capacity is 231, so each vehicle needs 8 slack bits, and the resulting QUBO has 112 variables.


In [5]:
def build_hard_capacity_assignment_qp(instance, num_vehicles, compactness_scale=1.0):
    qp = QuadraticProgram('cvrp_hard_capacity_assignment')
    add_assignment_variables(qp, instance['customers'], num_vehicles)
    add_each_customer_once_constraints(qp, instance['customers'], num_vehicles)

    capacity = int(instance['metadata']['CAPACITY'])
    for vehicle in range(num_vehicles):
        qp.linear_constraint(
            linear={
                assignment_variable_name(customer, vehicle): instance['demands'][customer]
                for customer in instance['customers']
            },
            sense='<=',
            rhs=capacity,
            name=f'capacity_vehicle_{vehicle}',
        )

    qp.minimize(quadratic=compactness_quadratic_terms(instance, num_vehicles, scale=compactness_scale))
    return qp


hard_assignment_qp = build_hard_capacity_assignment_qp(instance, parsed_vehicle_count)
hard_assignment_converter = QuadraticProgramToQubo()
hard_assignment_qubo = hard_assignment_converter.convert(hard_assignment_qp)

print('Hard-capacity assignment with slack')
print(f"Original variables: {hard_assignment_qp.get_num_vars()}")
print(f"Original constraints: {len(hard_assignment_qp.linear_constraints)}")
print(f"QUBO variables: {hard_assignment_qubo.get_num_vars()}")
print(f"Converter penalty: {hard_assignment_converter.penalty}")


Hard-capacity assignment with slack
Original variables: 80
Original constraints: 24
QUBO variables: 112
Converter penalty: 28997.0


## QUBO LP previews

These previews confirm the QUBO conversion happened. They are classical LP-string exports of the QUBO models, not quantum simulation.


In [6]:
PREVIEW_CHARS = 3000

print('Soft-capacity assignment QUBO preview')
soft_lp = soft_assignment_qubo.export_as_lp_string()
print(soft_lp[:PREVIEW_CHARS])
if len(soft_lp) > PREVIEW_CHARS:
    print()
    print(f"... truncated {len(soft_lp) - PREVIEW_CHARS} characters")

print()
print('Hard-capacity assignment QUBO preview')
hard_lp = hard_assignment_qubo.export_as_lp_string()
print(hard_lp[:PREVIEW_CHARS])
if len(hard_lp) > PREVIEW_CHARS:
    print()
    print(f"... truncated {len(hard_lp) - PREVIEW_CHARS} characters")


Soft-capacity assignment QUBO preview


/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_37180/529448217.py:4: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  soft_lp = soft_assignment_qubo.export_as_lp_string()


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: cvrp_soft_capacity_assignment

Minimize
 obj: - 9391034994 z_2_0 - 9391034994 z_2_1 - 9391034994 z_2_2
      - 9391034994 z_2_3 - 9379649994 z_3_0 - 9379649994 z_3_1
      - 9379649994 z_3_2 - 9379649994 z_3_3 - 9385441994 z_4_0
      - 9385441994 z_4_1 - 9385441994 z_4_2 - 9385441994 z_4_3
      - 9398209994 z_5_0 - 9398209994 z_5_1 - 9398209994 z_5_2
      - 9398209994 z_5_3 - 9362929994 z_6_0 - 9362929994 z_6_1
      - 9362929994 z_6_2 - 9362929994 z_6_3 - 9396601994 z_7_0
      - 9396601994 z_7_1 - 9396601994 z_7_2 - 9396601994 z_7_3
      - 9383329994 z_8_0 - 9383329994 z_8_1 - 9383329994 z_8_2
      - 9383329994 z_8_3 - 9378506994 z_9_0 - 9378506994 z_9_1
      - 9378506994 z_9_2 - 9378506994 z_9_3 - 9371689994 z_10_0
      - 9371689994 z_10_1 - 9371689994 z_10_2 - 9371689994 z_10_3
      - 9380774994 z_11_0 - 9380774994 z_11_1 - 9380774994 z_11_2
      - 9380774994 z_11_3 - 9370001994 z_12_0 - 9370001

/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_37180/529448217.py:12: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  hard_lp = hard_assignment_qubo.export_as_lp_string()


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: cvrp_hard_capacity_assignment

Minimize
 obj: - 1004804044 z_2_0 - 1004804044 z_2_1 - 1004804044 z_2_2
      - 1004804044 z_2_3 - 562715782 z_3_0 - 562715782 z_3_1 - 562715782 z_3_2
      - 562715782 z_3_3 - 777061606 z_4_0 - 777061606 z_4_1 - 777061606 z_4_2
      - 777061606 z_4_3 - 1339719394 z_5_0 - 1339719394 z_5_1 - 1339719394 z_5_2
      - 1339719394 z_5_3 - 26851222 z_6_0 - 26851222 z_6_1 - 26851222 z_6_2
      - 26851222 z_6_3 - 1259339710 z_7_0 - 1259339710 z_7_1 - 1259339710 z_7_2
      - 1259339710 z_7_3 - 696681922 z_8_0 - 696681922 z_8_1 - 696681922 z_8_2
      - 696681922 z_8_3 - 522525940 z_9_0 - 522525940 z_9_1 - 522525940 z_9_2
      - 522525940 z_9_3 - 294783502 z_10_0 - 294783502 z_10_1 - 294783502 z_10_2
      - 294783502 z_10_3 - 602905624 z_11_0 - 602905624 z_11_1
      - 602905624 z_11_2 - 602905624 z_11_3 - 241197046 z_12_0
      - 241197046 z_12_1 - 241197046 z_12_2 - 241197046 z_12